# Are NBFNet's Path Explanations Actually Faithful?

CAP 6938 AI Knowledge Systems, research track. Shadi Khodagholi, UCF.

NBFNet answers a knowledge graph query and explains its answer with a few paths through the graph. I test whether those paths are really what the model used: I delete or change edges, run the frozen model again, and see whether the answer moves.

This notebook is the whole project. It runs on Colab's free CPU runtime, and **Runtime > Run all** is safe: the long runs at the end are switched off unless you turn them on.

1. Setup (about 3 minutes)
2. Demo, two runs on different queries
3. Re-run part of the recorded benchmark and compare it with the saved results
4. Every figure in the report and the talk, redrawn from the saved results
5. The longer runs behind the report

## 1. Setup

Download the code from GitHub.

In [ ]:
import os, subprocess
REPO = 'https://github.com/shadi97kh/nbfnet-path-faithfulness.git'
WORK = '/content/nbfnet-path-faithfulness'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '-q', REPO, WORK], check=True)
os.chdir(WORK)
print('code is in', WORK)

Install PyTorch Geometric and ULTRA. ULTRA's repository includes the frozen checkpoint I use (`ultra_4g`), so nothing is trained here. If Colab has no `torch_scatter` build for its PyTorch version, a small pure-PyTorch copy from this repository is installed instead.

In [ ]:
import sys, torch
os.environ['CUDA_VISIBLE_DEVICES'] = ''    # everything runs on CPU
os.environ['PYTHONWARNINGS'] = 'ignore'
pip = [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check']
subprocess.run(pip + ['torch_geometric', 'easydict', 'ninja'], check=True)

def scatter_works():
    test = 'import torch_scatter; torch_scatter.scatter_add'
    return subprocess.run([sys.executable, '-c', test], capture_output=True).returncode == 0

major, minor = torch.__version__.split('+')[0].split('.')[:2]
if not scatter_works():
    wheels = f'https://data.pyg.org/whl/torch-{major}.{minor}.0+cpu.html'
    subprocess.run(pip + ['--only-binary=:all:', 'torch_scatter', '-f', wheels], capture_output=True)
if scatter_works():
    print('torch_scatter: prebuilt wheel')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torch_scatter'], capture_output=True)
    subprocess.run(pip + ['./torch_scatter_shim'], check=True)
    print('torch_scatter: no wheel for torch', torch.__version__, 'so using the pure-PyTorch copy')

if not os.path.exists('ultra/ckpts/ultra_4g.pth'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/DeepGraphLearning/ULTRA.git', 'ultra'], check=True)
    subprocess.run(['git', '-C', 'ultra', 'checkout', '-q', '427966ad8ed60420eef034063d44f3153addff90'], check=True)
print('torch', torch.__version__, '| ULTRA ready')

Load the frozen model and CoDEx-S, and run it once. The first run compiles ULTRA's CPU kernel, which takes about a minute, so the demo below starts right away.

In [ ]:
!python -m pathfaith.warmup --dataset CoDExSmall

## 2. Demo

**Run 1.** Five test queries on CoDEx-S, a graph the model never saw in training. For each query the demo prints the model's answer and NBFNet's own top paths. Then it changes the graph and runs the model again, once per line: it deletes those paths, deletes the same number of random edges as a control, deletes only the edge the paths share most, moves the last hop to another entity, and turns the main relation down to half and to zero. At the end it prints how many queries finished every step, and the full benchmark these five queries come from.

In [ ]:
!python -m pathfaith.demo --n 5 --seed 0 --color always

**Run 2.** Three different queries, drawn with another seed.

In [ ]:
!python -m pathfaith.demo --n 3 --seed 7 --no-benchmark --color always

Optional: the same demo on WordNet (WN18RR). The first time, it downloads the graph, which takes about a minute.

In [ ]:
!python -m pathfaith.demo --dataset WN18RR --n 2 --no-benchmark --color always

## 3. Re-run the recorded benchmark

The benchmark's queries come in a fixed order and the model gives the same output every time. So running the first few queries again should give back the numbers saved in `results/`, every field of every query.

The explanation as NBFNet releases it, on the first 5 CoDEx-S queries:

In [ ]:
!rm -rf check && mkdir -p check
!python -m pathfaith.run_experiment --dataset CoDExSmall --n 5 --pool 20 --topk 5 --seed 0 --path-controls 0 --out check/paths_codexsmall.jsonl
!python -m pathfaith.compare_runs --new check/paths_codexsmall.jsonl --ref results/paths_codexsmall_seed0.jsonl

The same with my corrected explanation:

In [ ]:
!python -m pathfaith.run_experiment --dataset CoDExSmall --n 5 --pool 20 --topk 5 --seed 0 --path-controls 0 --corrected --out check/paths_codexsmall_fixed.jsonl
!python -m pathfaith.compare_runs --new check/paths_codexsmall_fixed.jsonl --ref results/paths_codexsmall_fixed_seed0.jsonl

The bug. ULTRA's fast kernel, which makes the predictions, sends each message from `edge_index[1]` to `edge_index[0]`. When a gradient is needed, which is when NBFNet's `visualize()` runs, the code switches to PyTorch Geometric's generic layer, which sends it the other way. So the released explanation is the gradient of a different model. This prints the score gap between the two on 3 queries, and shows that running the explanation on the reversed graph closes it.

In [ ]:
!python -m pathfaith.direction_check --dataset CoDExSmall --n 3 --keep-scores 0 --out check/direction_codexsmall.jsonl

Relation channels. Each relation type has one vector in ULTRA that scales every message along edges of that type, a bit like a neuromodulator (Bull et al., NeurIPS 2025). I turn each channel's gain from 1 down to 0 and compare the effect with what the gradient predicts. First 3 queries, against the saved run:

In [ ]:
!python -m pathfaith.neuromod --dataset CoDExSmall --n 3 --out check/neuromod_codexsmall.jsonl
!python -m pathfaith.compare_runs --new check/neuromod_codexsmall.jsonl --ref results/neuromod_codexsmall_seed0.jsonl

Link prediction on the full CoDEx-S test set, with ULTRA's own evaluation code. The ULTRA paper reports an MRR of 0.472 for its model; the checkpoint here adds one more training graph to that model. About two minutes.

In [ ]:
!python -m pathfaith.run_baseline --dataset CoDExSmall --n 0 --batch-size 8 --out check/baseline_codexsmall.json 2>&1 | grep -E '"(n_triples|mrr|hits@1|hits@10)"'

Are 200 queries per graph enough? This reads the saved results only. Because the query order was fixed before anything was measured, the first n queries are a fair sample of size n, so each curve shows an estimate settling as queries are added.

In [ ]:
!python -m pathfaith.reliability
from pathfaith.figures import fig_convergence
from IPython.display import Image, display
fig_convergence('check/convergence.png', figsize=(11.07, 6.32), big=True)
display(Image('check/convergence.png'))

## 4. Every figure

This redraws every figure in the report and the talk from the saved results in `results/`, without running the model, in under a minute. The PDFs go to `report/figs/` and `presentation/figs/`, and the figures are shown below. Everything is also zipped into `figures.zip`, which you can download from the Files panel on the left.

In [ ]:
!python -m pathfaith.figures > /dev/null          # the PDFs, as the report and the talk use them
!python -m pathfaith.figures --png > /dev/null    # the same figures as PNG, to show here
!zip -qr figures.zip report/figs presentation/figs
import glob
from IPython.display import Image, Markdown, display
print(len(glob.glob('report/figs/*.pdf')), 'report figures and', len(glob.glob('presentation/figs/*.pdf')), 'slide figures written')

display(Markdown('### In the report'))
for name in ['direction', 'convergence', 'rank_vs_drop', 'robustness', 'dose_response']:
    display(Markdown(f'`report/figs/{name}.pdf`'))
    display(Image(f'report/figs/{name}.png', width=950))
display(Markdown('### More figures from the same results'))
for name in ['set_level', 'redundancy', 'released_vs_corrected', 'rank_heatmap', 'rank_heatmap_corrected', 'channels', 'ig']:
    display(Markdown(f'`report/figs/{name}.pdf`'))
    display(Image(f'report/figs/{name}.png', width=950))
display(Markdown('### Slide versions'))
for f in sorted(glob.glob('presentation/figs/*.png')):
    display(Markdown(f'`{f[:-4]}.pdf`'))
    display(Image(f, width=750))

## 5. The longer runs

Everything in `results/` came from the commands below. They are switched off because they take hours on Colab (about a day for all of them on a CPU), and `results/` already holds their output. Each run picks up where it stopped, so it only does work for queries that are missing.

The channel experiment on 200 CoDEx-S queries takes about an hour. It writes to `check/`, so it does not touch the saved results, and then compares the two:

In [ ]:
RUN_CHANNELS = False
if RUN_CHANNELS:
    !python -m pathfaith.neuromod --dataset CoDExSmall --n 200 --out check/neuromod_codexsmall.jsonl
    !python -m pathfaith.analyze_neuromod check/neuromod_codexsmall.jsonl
    !python -m pathfaith.compare_runs --new check/neuromod_codexsmall.jsonl --ref results/neuromod_codexsmall_seed0.jsonl
else:
    print('switched off: set RUN_CHANNELS = True to run it')

Every run behind the report, in order, writing to `results/`:

In [ ]:
RUN_EVERYTHING = False

base = '--pool 20 --topk 5 --seed 0 --path-controls 0'
runs = []
for ds, lc in [('WN18RR', 'wn18rr'), ('FB15k237', 'fb15k237'), ('CoDExSmall', 'codexsmall'), ('NELL23k', 'nell23k')]:
    f = f'results/paths_{lc}_seed0.jsonl'                      # the explanation as released
    runs += [f'run_experiment --dataset {ds} --n 200 {base} --out {f}', f'enrich --dataset {ds} --file {f}']
for ds, lc, n in [('WN18RR', 'wn18rr', 200), ('FB15k237', 'fb15k237', 100), ('CoDExSmall', 'codexsmall', 200), ('NELL23k', 'nell23k', 200)]:
    f = f'results/paths_{lc}_fixed_seed0.jsonl'                # the corrected explanation
    runs += [f'run_experiment --dataset {ds} --n {n} {base} --corrected --out {f}', f'enrich --dataset {ds} --file {f}']
for ds, lc in [('CoDExSmall', 'codexsmall'), ('NELL23k', 'nell23k')]:
    f = f'results/paths_{lc}_3g_fixed_seed0.jsonl'             # corrected, on the ULTRA paper's own checkpoint
    runs += [f'run_experiment --dataset {ds} --n 200 {base} --corrected --ckpt ultra_3g.pth --out {f}', f'enrich --dataset {ds} --file {f}']
runs += [f'neuromod --dataset {ds} --n {n}'                    # relation channels
         for ds, n in [('CoDExSmall', 200), ('NELL23k', 200), ('WN18RR', 100), ('FB15k237', 50)]]
runs += [f'ig --dataset {ds} --n {n}'                          # integrated gradients
         for ds, n in [('CoDExSmall', 200), ('NELL23k', 200), ('WN18RR', 100), ('FB15k237', 50)]]
runs += ['ig --dataset CoDExSmall --n 50 --steps 32 --out results/ig_codexsmall_steps32_seed0.jsonl',   # the step ablation
         'ig --dataset CoDExSmall --n 50 --steps 64 --out results/ig_codexsmall_steps64_seed0.jsonl',
         'ig --dataset CoDExSmall --n 30 --released',                                                 # with the released gradient
         'direction_check --dataset CoDExSmall --n 30',                                               # the bug
         'direction_check --dataset NELL23k --n 20']
runs += [f'run_baseline --dataset {ds} --ckpt {ck} --n {n} --batch-size 8 --out results/{out}'        # link prediction
         for ds, ck, n, out in [('CoDExSmall', 'ultra_4g.pth', 0, 'baseline_codexsmall.json'),
                                ('WN18RR', 'ultra_4g.pth', 0, 'baseline_wn18rr_full.json'),
                                ('FB15k237', 'ultra_4g.pth', 2000, 'baseline_fb15k237_2000.json'),
                                ('NELL23k', 'ultra_4g.pth', 0, 'baseline_nell23k_full.json'),
                                ('CoDExSmall', 'ultra_3g.pth', 0, 'baseline3g_codexsmall_full.json'),
                                ('NELL23k', 'ultra_3g.pth', 0, 'baseline3g_nell23k_full.json')]]
runs += ['reliability', 'figures']                             # the reliability check and every figure

if RUN_EVERYTHING:
    for cmd in runs:
        print('>>', cmd, flush=True)
        !python -m pathfaith.{cmd}
else:
    print(len(runs), 'runs, switched off: set RUN_EVERYTHING = True to run them')